# P5-7 · GĐ 7 · Đóng gói sản phẩm

Toàn bộ code của bài, chạy từ trên xuống như trên trang bài học. Kết quả đã chạy sẵn được giữ trong notebook; bấm **Runtime → Run all** để chạy lại. Ô có dòng đầu `# Cần GPU` thì đổi **Runtime → Change runtime type → T4 GPU**; ô cần API key thì đặt khóa trong Colab Secrets.

In [ ]:
!pip install -q diffusers gradio

## Lý thuyết

In [ ]:
# Cần GPU và pipe đã nạp ở P5.4: chạy trên Colab (launch mở giao diện web)
import gradio as gr

def generate(prompt, steps, seed):
    # gọi pipeline đã nạp sẵn ở cấp module; trả về ảnh PIL
    return pipe(prompt, num_inference_steps=int(steps),
                generator=torch.Generator("cuda").manual_seed(int(seed))).images[0]

demo = gr.Interface(fn=generate,
                    inputs=[gr.Textbox(label="Mô tả"), gr.Slider(10, 40, value=25, step=1, label="Số bước"),
                            gr.Number(value=0, label="Seed")],
                    outputs=gr.Image(label="Ảnh"))
demo.queue(max_size=20).launch()

## Bài tập 5: App Gradio {code}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
# Cần GPU, mạng và LoRA đã train: chạy trên Colab
import gradio as gr, torch
from diffusers import StableDiffusionPipeline, DPMSolverMultistepScheduler

pipe = StableDiffusionPipeline.from_pretrained("stable-diffusion-v1-5/stable-diffusion-v1-5", torch_dtype=torch.float16).to("cuda")
pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)
pipe.load_lora_weights("lora-tthstyle")

def generate(prompt, negative, steps, guidance, lora_scale, seed):
    if not prompt.strip():
        raise gr.Error("Hãy nhập mô tả ảnh.")
    g = torch.Generator("cuda").manual_seed(int(seed))
    return pipe(prompt[:300], negative_prompt=negative[:300], num_inference_steps=int(steps), guidance_scale=guidance,
                generator=g, cross_attention_kwargs={"scale": lora_scale}).images[0]

with gr.Blocks(title="Tạo ảnh sản phẩm nội thất") as demo:
    gr.Markdown("## Tạo ảnh sản phẩm nội thất\nStable Diffusion 1.5 + LoRA phong cách ảnh sản phẩm.")
    with gr.Row():
        with gr.Column():
            prompt = gr.Textbox(label="Mô tả (tiếng Anh)", lines=2)
            negative = gr.Textbox(label="Không muốn có", value="blurry, text, watermark")
            steps = gr.Slider(10, 40, value=25, step=1, label="Số bước")
            guidance = gr.Slider(1, 12, value=7, step=0.5, label="Guidance")
            lora_scale = gr.Slider(0, 1, value=0.8, step=0.05, label="Độ mạnh phong cách")
            seed = gr.Number(value=42, precision=0, label="Seed")
            btn = gr.Button("Tạo ảnh", variant="primary")
        out = gr.Image(label="Kết quả")
    gr.Examples([["a vanity table with round mirror, tthstyle product photo"],
                 ["a rattan armchair, tthstyle product photo"],
                 ["a ceramic table lamp, tthstyle product photo"]], inputs=[prompt])
    btn.click(generate, [prompt, negative, steps, guidance, lora_scale, seed], out)

demo.queue(max_size=20).launch()